# Extraction Validation System

This notebook validates the LLM extraction pipeline by:
1. Fetching papers from existing ORKG comparison (R1364660)
2. Re-extracting data using the pipeline
3. Comparing extracted data with existing ORKG data
4. Generating detailed reports and accuracy metrics

## Section 1: Setup and Imports


In [ ]:
import sys
import os
from pathlib import Path
import json
from datetime import datetime
from typing import Dict, List, Any, Optional

# Add src to path
sys.path.insert(0, str(Path.cwd() / 'src'))

# Import pipeline components
from src.orkg_client import ORKGClient
from src.paper_fetcher import PaperFetcher
from src.pdf_parser import PDFParser
from src.llm_extractor import LLMExtractor
from src.template_mapper import TemplateMapper
from src.validation_comparator import ValidationComparator

# Import visualization libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown
from tqdm import tqdm

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

print("✅ All imports successful!")


In [ ]:
# Configuration
COMPARISON_ID = "R1364660"  # ORKG comparison ID
ORKG_HOST = "sandbox"  # or "production"
GEMINI_MODEL = "gemini-2.5-flash"

# Get API keys from environment
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
ORKG_EMAIL = os.getenv("ORKG_EMAIL")
ORKG_PASSWORD = os.getenv("ORKG_PASSWORD")

# Output directory
OUTPUT_DIR = Path("data/validation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration loaded:")
print(f"  - Comparison ID: {COMPARISON_ID}")
print(f"  - ORKG Host: {ORKG_HOST}")
print(f"  - Gemini Model: {GEMINI_MODEL}")
print(f"  - Output Directory: {OUTPUT_DIR}")


In [ ]:
# Initialize pipeline components
orkg_client = ORKGClient(
    host=ORKG_HOST,
    email=ORKG_EMAIL,
    password=ORKG_PASSWORD
)

paper_fetcher = PaperFetcher(download_dir="data/papers")
pdf_parser = PDFParser(method="pdfplumber")

if GEMINI_API_KEY:
    llm_extractor = LLMExtractor(
        api_key=GEMINI_API_KEY,
        model=GEMINI_MODEL,
        temperature=0.0,
        max_tokens=4000
    )
else:
    raise ValueError("GEMINI_API_KEY not found in environment!")

template_mapper = TemplateMapper(template_id="R609825")
validation_comparator = ValidationComparator(similarity_threshold=0.8)

print("✅ All components initialized!")


## Section 2: Fetch Comparison Data


In [ ]:
# Fetch comparison
print(f"Fetching comparison {COMPARISON_ID}...")
comparison = orkg_client.get_comparison(COMPARISON_ID)

if comparison:
    print(f"✅ Comparison found: {comparison.get('label', 'Unknown')}")
    print(f"   ID: {comparison.get('id')}")
    print(f"   Description: {comparison.get('description', 'N/A')}")
else:
    print("❌ Comparison not found!")


In [ ]:
# Extract paper metadata from comparison
print("Extracting paper metadata from contributions...")
papers_data = orkg_client.extract_paper_metadata_from_comparison(COMPARISON_ID)

print(f"\n✅ Found {len(papers_data)} papers/contributions")

# Display summary
if papers_data:
    papers_df = pd.DataFrame([
        {
            'Contribution Label': p.get('contribution_label', 'N/A'),
            'Paper ID': p.get('paper_id', 'N/A'),
            'ArXiv ID': p.get('arxiv_id', 'N/A'),
            'Paper Title': p.get('paper_title', 'N/A')[:50] + '...' if p.get('paper_title') and len(p.get('paper_title', '')) > 50 else p.get('paper_title', 'N/A')
        }
        for p in papers_data
    ])
    display(papers_df)


## Section 3: Run Extraction Pipeline


In [ ]:
# Filter papers with ArXiv IDs
papers_with_arxiv = [p for p in papers_data if p.get('arxiv_id')]

print(f"Found {len(papers_with_arxiv)} papers with ArXiv IDs")
print(f"Total papers: {len(papers_data)}")

# Optionally limit to first N papers for testing
MAX_PAPERS = None  # Set to a number (e.g., 3) to limit, or None for all
if MAX_PAPERS:
    papers_with_arxiv = papers_with_arxiv[:MAX_PAPERS]
    print(f"Limited to first {MAX_PAPERS} papers for testing")


In [ ]:
# Run extraction pipeline for each paper
extraction_results = []
failed_papers = []

for i, paper_data in enumerate(tqdm(papers_with_arxiv, desc="Processing papers"), 1):
    arxiv_id = paper_data.get('arxiv_id')
    paper_title = paper_data.get('paper_title', 'Unknown')
    contribution_id = paper_data.get('contribution_id')
    
    print(f"\n{'='*80}")
    print(f"Paper {i}/{len(papers_with_arxiv)}: {paper_title}")
    print(f"ArXiv ID: {arxiv_id}")
    print(f"{'='*80}")
    
    try:
        # Step 1: Fetch paper metadata
        print("  📄 Fetching paper metadata...")
        metadata = paper_fetcher.fetch_paper_metadata(arxiv_id)
        if not metadata:
            print(f"  ❌ Failed to fetch metadata for {arxiv_id}")
            failed_papers.append({'arxiv_id': arxiv_id, 'error': 'metadata_fetch_failed'})
            continue
        
        # Step 2: Download PDF
        print("  📥 Downloading PDF...")
        pdf_path = paper_fetcher.download_pdf(arxiv_id)
        if not pdf_path:
            print(f"  ❌ Failed to download PDF for {arxiv_id}")
            failed_papers.append({'arxiv_id': arxiv_id, 'error': 'pdf_download_failed'})
            continue
        
        # Step 3: Parse PDF
        print("  📖 Parsing PDF...")
        paper_text = pdf_parser.parse(str(pdf_path))
        if not paper_text or len(paper_text) < 100:
            print(f"  ❌ Failed to parse PDF or text too short")
            failed_papers.append({'arxiv_id': arxiv_id, 'error': 'pdf_parse_failed'})
            continue
        
        print(f"  ✅ Parsed {len(paper_text)} characters")
        
        # Step 4: Extract with LLM
        print("  🤖 Extracting data with LLM...")
        extraction_result = llm_extractor.extract(paper_text, metadata)
        if not extraction_result or not extraction_result.models:
            print(f"  ❌ Failed to extract data")
            failed_papers.append({'arxiv_id': arxiv_id, 'error': 'extraction_failed'})
            continue
        
        print(f"  ✅ Extracted {len(extraction_result.models)} model(s)")
        
        # Step 5: Map to ORKG format
        print("  🔄 Mapping to ORKG format...")
        mapped_result = template_mapper.map_extraction_result(extraction_result)
        
        # Store results
        extraction_results.append({
            'paper_data': paper_data,
            'metadata': metadata,
            'extraction_result': extraction_result,
            'mapped_result': mapped_result,
            'paper_text_length': len(paper_text)
        })
        
        print(f"  ✅ Successfully processed paper {i}")
        
    except Exception as e:
        print(f"  ❌ Error processing paper: {e}")
        import traceback
        traceback.print_exc()
        failed_papers.append({'arxiv_id': arxiv_id, 'error': str(e)})

print(f"\n{'='*80}")
print(f"Extraction Summary:")
print(f"  ✅ Successfully processed: {len(extraction_results)}")
print(f"  ❌ Failed: {len(failed_papers)}")
print(f"{'='*80}")


## Section 4: Comparison Analysis


In [ ]:
# Compare extracted data with existing ORKG data
comparisons = []

for result in extraction_results:
    paper_data = result['paper_data']
    mapped_result = result['mapped_result']
    
    # Get the first model from extraction (assuming one model per paper)
    if mapped_result.get('contributions'):
        extracted_contrib = mapped_result['contributions'][0]
        
        # Create existing contribution structure from paper_data
        existing_contrib = {
            'label': paper_data.get('contribution_label', 'Unknown'),
            'properties': []
        }
        
        # Convert existing properties to contribution format
        # Map property IDs to field names (reverse mapping)
        reverse_mapping = {v: k for k, v in template_mapper.field_mapping.items()}
        
        for prop_id, value in paper_data.get('existing_properties', {}).items():
            field_name = reverse_mapping.get(prop_id, prop_id)
            existing_contrib['properties'].append({
                'label': field_name,
                'property': prop_id,
                'value': value
            })
        
        # Perform comparison
        comparison = validation_comparator.compare_contributions(
            extracted_contrib,
            existing_contrib
        )
        
        comparison['paper_title'] = paper_data.get('paper_title', 'Unknown')
        comparison['arxiv_id'] = paper_data.get('arxiv_id', 'Unknown')
        
        comparisons.append(comparison)

print(f"✅ Completed {len(comparisons)} comparisons")


In [ ]:
# Display comparison summary table
if comparisons:
    summary_data = []
    for comp in comparisons:
        summary = comp.get('summary', {})
        summary_data.append({
            'Paper': comp.get('paper_title', 'Unknown')[:50],
            'ArXiv ID': comp.get('arxiv_id', 'N/A'),
            'Accuracy': f"{summary.get('accuracy', 0.0):.1%}",
            'Matches': summary.get('matches', 0),
            'Partial': summary.get('partial_matches', 0),
            'Mismatches': summary.get('mismatches', 0),
            'Missing': summary.get('missing_in_extracted', 0)
        })
    
    summary_df = pd.DataFrame(summary_data)
    display(summary_df)


In [ ]:
# Detailed field-by-field comparison for first paper
if comparisons:
    first_comp = comparisons[0]
    
    print(f"Detailed Comparison: {first_comp.get('paper_title', 'Unknown')}")
    print("="*80)
    
    field_data = []
    for field_name, field_info in first_comp.get('fields', {}).items():
        status = field_info.get('status', 'unknown')
        extracted = str(field_info.get('extracted', ''))[:50]
        existing = str(field_info.get('existing', ''))[:50]
        similarity = field_info.get('similarity', 0.0)
        
        field_data.append({
            'Field': field_name,
            'Status': status,
            'Extracted': extracted,
            'Existing': existing,
            'Similarity': f"{similarity:.2f}"
        })
    
    field_df = pd.DataFrame(field_data)
    
    # Color code by status
    def color_status(val):
        if val == 'match':
            return 'background-color: #d4edda'
        elif val == 'partial':
            return 'background-color: #fff3cd'
        elif val == 'mismatch':
            return 'background-color: #f8d7da'
        else:
            return ''
    
    styled_df = field_df.style.applymap(color_status, subset=['Status'])
    display(styled_df)


## Section 5: Accuracy Metrics


In [ ]:
# Calculate overall accuracy metrics
metrics = validation_comparator.calculate_accuracy_metrics(comparisons)

print("="*80)
print("ACCURACY METRICS")
print("="*80)
print(f"\nOverall Accuracy: {metrics.get('overall_accuracy', 0.0):.1%}")
print(f"Total Papers Validated: {metrics.get('total_papers', 0)}")

# Display field accuracy
field_accuracy = metrics.get('field_accuracy', {})
if field_accuracy:
    print("\nField-by-Field Accuracy:")
    print("-"*80)
    
    field_acc_data = []
    for field_name, acc_data in field_accuracy.items():
        field_acc_data.append({
            'Field': field_name,
            'Accuracy': f"{acc_data.get('accuracy', 0.0):.1%}",
            'Matches': acc_data.get('matches', 0),
            'Partial': acc_data.get('partial_matches', 0),
            'Mismatches': acc_data.get('mismatches', 0),
            'Missing': acc_data.get('missing_extracted', 0),
            'Avg Similarity': f"{acc_data.get('avg_similarity', 0.0):.2f}"
        })
    
    field_acc_df = pd.DataFrame(field_acc_data)
    field_acc_df = field_acc_df.sort_values('Accuracy', ascending=False)
    display(field_acc_df)


In [ ]:
# Visualize accuracy metrics
if field_accuracy:
    # Bar chart of field accuracies
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Field accuracy chart
    field_names = list(field_accuracy.keys())
    accuracies = [field_accuracy[f].get('accuracy', 0.0) for f in field_names]
    
    axes[0].barh(field_names, accuracies, color='steelblue')
    axes[0].set_xlabel('Accuracy')
    axes[0].set_title('Field Accuracy')
    axes[0].set_xlim(0, 1)
    axes[0].axvline(x=0.8, color='red', linestyle='--', alpha=0.5, label='80% threshold')
    axes[0].legend()
    
    # Match vs Mismatch chart
    matches = [field_accuracy[f].get('matches', 0) for f in field_names]
    mismatches = [field_accuracy[f].get('mismatches', 0) for f in field_names]
    
    x = range(len(field_names))
    width = 0.35
    axes[1].bar([i - width/2 for i in x], matches, width, label='Matches', color='green', alpha=0.7)
    axes[1].bar([i + width/2 for i in x], mismatches, width, label='Mismatches', color='red', alpha=0.7)
    axes[1].set_xlabel('Fields')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Matches vs Mismatches by Field')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(field_names, rotation=45, ha='right')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()


In [ ]:
# Heatmap of accuracy by paper and field
if comparisons and len(comparisons) > 1:
    # Create matrix: papers x fields
    paper_titles = [c.get('paper_title', f"Paper {i}")[:30] for i, c in enumerate(comparisons)]
    all_fields = set()
    for comp in comparisons:
        all_fields.update(comp.get('fields', {}).keys())
    
    all_fields = sorted(list(all_fields))
    
    # Build similarity matrix
    similarity_matrix = []
    for comp in comparisons:
        row = []
        for field in all_fields:
            field_data = comp.get('fields', {}).get(field, {})
            similarity = field_data.get('similarity', 0.0)
            row.append(similarity)
        similarity_matrix.append(row)
    
    # Create heatmap
    plt.figure(figsize=(max(12, len(all_fields) * 0.8), max(6, len(paper_titles) * 0.5)))
    sns.heatmap(
        similarity_matrix,
        xticklabels=all_fields,
        yticklabels=paper_titles,
        annot=True,
        fmt='.2f',
        cmap='RdYlGn',
        vmin=0,
        vmax=1,
        cbar_kws={'label': 'Similarity Score'}
    )
    plt.title('Field Similarity Heatmap (Papers × Fields)')
    plt.xlabel('Fields')
    plt.ylabel('Papers')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


## Section 6: Generate Reports


In [ ]:
# Generate HTML report
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
report_path = OUTPUT_DIR / f"validation_report_{timestamp}.html"

print(f"Generating HTML report...")
html_report = validation_comparator.generate_comparison_report(
    comparisons,
    metrics,
    output_path=str(report_path)
)

print(f"✅ Report saved to: {report_path}")

# Display report in notebook
display(HTML(html_report))


In [ ]:
# Export metrics to JSON
metrics_path = OUTPUT_DIR / f"metrics_{timestamp}.json"
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False, default=str)

print(f"✅ Metrics saved to: {metrics_path}")


In [ ]:
# Export detailed comparison to CSV
comparison_data = []
for comp in comparisons:
    for field_name, field_info in comp.get('fields', {}).items():
        comparison_data.append({
            'Paper': comp.get('paper_title', 'Unknown'),
            'ArXiv ID': comp.get('arxiv_id', 'N/A'),
            'Field': field_name,
            'Status': field_info.get('status', 'unknown'),
            'Extracted': str(field_info.get('extracted', '')),
            'Existing': str(field_info.get('existing', '')),
            'Similarity': field_info.get('similarity', 0.0),
            'Match': field_info.get('match', False)
        })

if comparison_data:
    comparison_df = pd.DataFrame(comparison_data)
    csv_path = OUTPUT_DIR / f"detailed_comparison_{timestamp}.csv"
    comparison_df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"✅ Detailed comparison saved to: {csv_path}")
    
    # Display sample
    display(comparison_df.head(20))


## Summary

### Key Findings

Review the metrics and comparison results above to see:
- Overall accuracy across all papers
- Field-by-field performance
- Best and worst performing fields
- Detailed comparisons for each paper

### Recommendations

1. Review extraction prompts for low-performing fields
2. Check if field mappings are correct
3. Consider adding post-processing for common errors
4. Validate required fields are being extracted consistently
5. Review the generated HTML report for detailed analysis
